# Data Preparation Pipeline

This notebook prepares and processes raw data for ASEAN Green Bonds research using the `asean_green_bonds` package.

**Pipeline:**
1. Load raw data (panel, ESG, market, green bonds)
2. Merge datasets with proper identifiers
3. Filter to ASEAN firms and valid years
4. Handle missing values and merge industry data
5. Feature engineering (log transforms, indicators)
6. Save processed data for analysis

In [1]:
# Import ASEAN Green Bonds package
from asean_green_bonds import data, config
import pandas as pd
import numpy as np

print(f'Package imported successfully')
print(f'Data directory: {config.DATA_DIR}')
print(f'Output directory: {config.PROCESSED_DATA_DIR}')

Package imported successfully
Data directory: /Users/bunnypro/Projects/refinitiv-search/data
Output directory: /Users/bunnypro/Projects/refinitiv-search/processed_data


In [2]:
# Load all raw data sources
print('Loading raw data...')

panel_df = data.load_raw_panel_data()
esg_df = data.load_esg_panel_data()
market_df = data.load_market_data()
gb_df = data.load_green_bonds_data(asean_only=True)
series_df = data.load_series_data()

print(f'Panel data: {panel_df.shape}')
print(f'ESG data: {esg_df.shape}')
print(f'Market data: {market_df.shape}')
print(f'Green bonds: {gb_df.shape}')
print(f'Series data: {series_df.shape}')

Loading raw data...
Panel data: (55392, 26)
ESG data: (35240, 10)
Market data: (5126, 4)
Green bonds: (328, 4)
Series data: (4962, 2)


In [3]:
# Merge financial and ESG data
print('Merging datasets...')

merged_df = data.merge_panel_data(panel_df, esg_df, market_df)
print(f'Merged financial + ESG: {merged_df.shape}')

# Add green bonds indicators
merged_df = data.merge_green_bonds(merged_df, gb_df, market_df)
print(f'After green bonds merge: {merged_df.shape}')

# Add industry classification
merged_df = data.merge_industry_data(merged_df, series_df)
print(f'After industry merge: {merged_df.shape}')

Merging datasets...
Merged financial + ESG: (47007, 33)
After green bonds merge: (47007, 42)
After industry merge: (47007, 43)


In [13]:
# === FINAL SAMPLE PERIOD CONTROL ===
# Define the target range for analysis
# We keep one extra year at the start (min_year - 1) to allow for 1-year lag (L1) calculation
START_YEAR = config.TIME_PERIODS['pre_treatment_start']  # e.g., 2015
END_YEAR = config.TIME_PERIODS['analysis_end']          # e.g., 2024

print(f'Final Analysis Period: {START_YEAR} to {END_YEAR}')

# Filter to ASEAN firms and valid years
print('\nFiltering data...')

merged_df = data.filter_asean_firms_and_years(
    merged_df,
    min_year=START_YEAR,
    max_year=END_YEAR
)
print(f'After ASEAN & year filter: {merged_df.shape}')

# Handle missing values
merged_df = data.handle_missing_values(
    merged_df,
    firm_col='ric',
    forward_fill_cols=[
        'total_assets', 'total_debt', 'long_term_debt',
        'market_capitalization', 'employees'
    ],
    min_years_per_firm=3
)
print(f'After missing value handling: {merged_df.shape}')

Final Analysis Period: 2020 to 2024

Filtering data...
After ASEAN & year filter: (19978, 65)
After missing value handling: (19978, 65)


## Survivorship Bias Handling

Check for and handle potential survivorship bias in the panel data.

In [14]:
# ============================================================
# SURVIVORSHIP BIAS HANDLING
# ============================================================

from asean_green_bonds.data import prepare_analysis_sample, filter_survived_firms
from asean_green_bonds.config import SURVIVORSHIP_CONFIG

print('\n' + '='*60)
print('SURVIVORSHIP BIAS ANALYSIS')
print('='*60)

recent_years = SURVIVORSHIP_CONFIG['recent_years']
early_years = SURVIVORSHIP_CONFIG.get('early_years', [2015, 2016, 2017])
print(f'\nChecking firm presence in recent years: {recent_years}')

all_firms = merged_df['ric'].nunique()
survived_df = filter_survived_firms(merged_df, firm_col='ric', recent_years=recent_years)
survived_firms = survived_df['ric'].nunique()
dropped_firms = all_firms - survived_firms

print(f'Total firms in panel: {all_firms}')
print(f'Firms with recent data (survived): {survived_firms}')
print(f'Firms without recent data (potential attrition): {dropped_firms} ({100*dropped_firms/all_firms:.1f}%)')

# Backward-compatible default remains available: 'ignore'
survivorship_mode = SURVIVORSHIP_CONFIG.get('mode', 'exclude')  # ignore | exclude | weight

merged_df = prepare_analysis_sample(
    merged_df,
    survivorship_mode=survivorship_mode,
    firm_col='ric',
    time_col='Year',
    recent_years=recent_years,
    early_years=early_years,
)

print(f'\nSurvivorship mode: {survivorship_mode}')
print(f'Final panel size: {merged_df.shape}')

if survivorship_mode == 'weight' and 'survivorship_weight' in merged_df.columns:
    print('Weight statistics:')
    print(f'  Mean: {merged_df["survivorship_weight"].mean():.3f}')
    print(f'  Range: [{merged_df["survivorship_weight"].min():.3f}, {merged_df["survivorship_weight"].max():.3f}]')

if 'share_certified_proceeds' in merged_df.columns and 'self_labeled_share' in merged_df.columns:
    issuance_mask = merged_df['green_bond_issue'].fillna(0) > 0
    if issuance_mask.any():
        print('\nCertification intensity (issuance-years):')
        print(f'  Mean certified share: {merged_df.loc[issuance_mask, "share_certified_proceeds"].mean():.3f}')
        print(f'  Mean self-labeled share: {merged_df.loc[issuance_mask, "self_labeled_share"].mean():.3f}')



SURVIVORSHIP BIAS ANALYSIS

Checking firm presence in recent years: [2023, 2024, 2025]
Total firms in panel: 4004
Firms with recent data (survived): 4004
Firms without recent data (potential attrition): 0 (0.0%)

Survivorship mode: exclude
Final panel size: (19978, 65)

Certification intensity (issuance-years):
  Mean certified share: 1.000
  Mean self-labeled share: 0.000


In [15]:
# STEP 1: Create financial ratio features
print('\nEngineering financial ratios...')

# Create financial ratios from raw metrics:
# - Firm_Size = ln(total_assets)
# - Leverage = total_debt / total_assets
# - Asset_Turnover = net_sales / total_assets
# - Capital_Intensity = total_assets / net_sales
# - Cash_Ratio = cash / current_liabilities
merged_df = data.create_financial_ratios(merged_df)
print(f'  ✓ Financial ratios created: Firm_Size, Leverage, Asset_Turnover, Capital_Intensity, Cash_Ratio')

# STEP 2: Create lagged features for control variables
print('\nCreating lagged features...')
merged_df = data.create_lagged_features(
    merged_df,
    firm_col='ric',
    vars_to_lag=['Firm_Size', 'Leverage', 'Asset_Turnover', 'Cash_Ratio', 'Capital_Intensity', 
                 'return_on_assets', 'Tobin_Q', 'esg_score'],
    lags=[1]
)
print(f'  ✓ Lagged features created (L1_* variables)')

# STEP 3: Create log-transformed features
print('\nCreating log-transformed features...')
merged_df = data.create_log_features(
    merged_df,
    cols_to_log=['total_assets', 'employees', 'net_sales_or_revenues']
)
print(f'  ✓ Log features created')

# STEP 4: Normalize percentages
print('\nNormalizing percentages...')
merged_df = data.normalize_percentages(
    merged_df,
    pct_cols=['return_on_assets', 'return_on_equity_total']
)
print(f'  ✓ Percentages normalized')

# STEP 5: Winsorize outliers
print('\nWinsorizing outliers...')
merged_df = data.winsorize_outliers(merged_df)
print(f'  ✓ Outliers winsorized')

# STEP 6: Encode categorical features
print('\nEncoding categorical features...')
merged_df = data.encode_categorical_features(merged_df)
print(f'  ✓ Categorical features encoded')

# Verify control variables are available
from asean_green_bonds.config import CONTROL_VARIABLES
print(f'\nControl Variables Status:')
available_controls = [v for v in CONTROL_VARIABLES if v in merged_df.columns]
print(f'  ✓ {len(available_controls)}/{len(CONTROL_VARIABLES)} control variables ready')
for var in CONTROL_VARIABLES:
    status = '✓' if var in merged_df.columns else '✗'
    non_null = merged_df[var].notna().sum() if var in merged_df.columns else 0
    pct = 100 * non_null / len(merged_df) if var in merged_df.columns else 0
    print(f'    {status} {var}: {non_null} non-null ({pct:.1f}%)')


Engineering financial ratios...
  ✓ Financial ratios created: Firm_Size, Leverage, Asset_Turnover, Capital_Intensity, Cash_Ratio

Creating lagged features...
  ✓ Lagged features created (L1_* variables)

Creating log-transformed features...
  ✓ Log features created

Normalizing percentages...
  ✓ Percentages normalized

Winsorizing outliers...
  ✓ Outliers winsorized

Encoding categorical features...
  ✓ Categorical features encoded

Control Variables Status:
  ✓ 5/5 control variables ready
    ✓ L1_Firm_Size: 15942 non-null (79.8%)
    ✓ L1_Leverage: 15938 non-null (79.8%)
    ✓ L1_Asset_Turnover: 14892 non-null (74.5%)
    ✓ L1_Capital_Intensity: 14750 non-null (73.8%)
    ✓ L1_Cash_Ratio: 12335 non-null (61.7%)


/Users/bunnypro/miniconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [7]:
# === ENGINEER PSM ATTRIBUTES ===
print('\n' + '='*60)
print('ENGINEERING PSM ATTRIBUTES FOR CAUSAL ANALYSIS')
print('='*60)

from asean_green_bonds.data.feature_engineering import engineer_psm_attributes

# Load raw green bonds data
gb_auth = pd.read_csv('data/green_bonds_with_authenticity_score.csv')
gb_raw = pd.read_csv('data/green-bonds.csv')

# Engineer 4 PSM attributes:
# 1. has_green_framework (from issuer verification)
# 2. asset_tangibility (sector-based proxy)
# 3. issuer_track_record (from issuer verification)
# 4. prior_green_bonds (from issuance history)
gb_engineered, psm_metadata = engineer_psm_attributes(gb_auth, gb_raw, verbose=True)

# Save engineered green bonds dataset
gb_output_path = 'data/green_bonds_with_psm_features.csv'
gb_engineered.to_csv(gb_output_path, index=False)

print(f'\n✓ Engineered dataset shape: {gb_engineered.shape}')
print(f'✓ Saved to: {gb_output_path}')

# Verify all attributes
required_psm = ['has_green_framework', 'asset_tangibility', 'issuer_track_record', 'prior_green_bonds']
available_psm = [v for v in required_psm if v in gb_engineered.columns]
print(f'\n✓ PSM Variables: {len(available_psm)}/{len(required_psm)} ready')
for var in required_psm:
    status = '✓' if var in gb_engineered.columns else '✗'
    print(f'  {status} {var}')


ENGINEERING PSM ATTRIBUTES FOR CAUSAL ANALYSIS
ENGINEERING MISSING PSM ATTRIBUTES

Step 1: Standardizing existing attributes (lowercase)...
  ✓ has_green_framework: 328 issuers
  ✓ issuer_track_record: min=0, max=65

Step 2: Engineering prior_green_bonds from issuance history...
  ✓ prior_green_bonds: 0 to 65
    - First-time issuers: 70
    - Repeat issuers: 263

Step 3: Estimating asset_tangibility by sector...
  ✓ asset_tangibility: 0.626 (mean)
    Range: 0.55 - 0.85
    Sectors mapped: 139
    Using fallback: 194

VERIFICATION: All PSM Attributes Present
  ✓ has_green_framework            - 333/333 non-null
  ✓ asset_tangibility              - 333/333 non-null
  ✓ issuer_track_record            - 333/333 non-null
  ✓ prior_green_bonds              - 333/333 non-null

✅ SUCCESS: All 4 PSM attributes engineered!

✓ Engineered dataset shape: (333, 24)
✓ Saved to: data/green_bonds_with_psm_features.csv

✓ PSM Variables: 4/4 ready
  ✓ has_green_framework
  ✓ asset_tangibility
  ✓ issu

In [16]:
# === MERGE PSM ATTRIBUTES INTO FINAL PANEL ===
from asean_green_bonds.data.feature_engineering import merge_psm_into_panel, normalize_psm_attributes

# Load engineered green bonds
gb_engineered = pd.read_csv('data/green_bonds_with_psm_features.csv')

# Merge PSM features into the panel
merged_df = merge_psm_into_panel(merged_df, gb_engineered, market_df, verbose=True)

# Normalize attribute names to lowercase
merged_df = normalize_psm_attributes(merged_df, verbose=False)

# Verify PSM columns
psm_check = ['has_green_framework', 'asset_tangibility', 'issuer_track_record', 'prior_green_bonds']
present = [col for col in psm_check if col in merged_df.columns]
print(f'\n✓ PSM attributes in final panel: {len(present)}/{len(psm_check)}')

# Summary for green bond issuers
gb_panel = merged_df[merged_df['green_bond_issue'] == 1]
if len(gb_panel) > 0:
    print(f'\n✓ Green bond issuer coverage:')
    print(f'  Total GB records: {len(gb_panel)}')
    for col in psm_check:
        if col in gb_panel.columns:
            if col == 'asset_tangibility':
                has_value = (gb_panel[col] != 0.55).sum()
            else:
                has_value = (gb_panel[col] > 0).sum()
            print(f'  {col:35s}: {has_value}/{len(gb_panel)} with non-default')

MERGING PSM ATTRIBUTES INTO FINAL PANEL

Step 1: org_permid already present: 19963/19978

Step 2: Aggregating PSM attributes by issuer...
  ✓ Aggregated PSM for 64 unique issuers

Step 3: Initializing PSM attributes with defaults...

Step 4: Merging PSM features from green bond issuers...
  ✓ Merged into panel: (19978, 65)

Step 5: Verification
  ✓ has_green_framework                 - 19978/19978 non-null
  ✓ asset_tangibility                   - 19978/19978 non-null
  ✓ issuer_track_record                 - 19978/19978 non-null
  ✓ prior_green_bonds                   - 19978/19978 non-null

✓ PSM attributes in final panel: 4/4

✓ Green bond issuer coverage:
  Total GB records: 43
  has_green_framework                : 43/43 with non-default
  asset_tangibility                  : 30/43 with non-default
  issuer_track_record                : 30/43 with non-default
  prior_green_bonds                  : 30/43 with non-default


## ESG Data Coverage Analysis

Analyze ESG data availability across countries and its impact on sample composition.

In [17]:
# ============================================================
# ESG DATA COVERAGE ANALYSIS
# ============================================================

from asean_green_bonds.authenticity import get_esg_coverage_by_country

print('\n' + '='*60)
print('ESG DATA COVERAGE BY COUNTRY')
print('='*60)

# Load ESG and green bonds data for coverage analysis
try:
    esg_coverage = get_esg_coverage_by_country(
        esg_df,
        gb_df if 'gb_df' in dir() else data.load_green_bonds_data()
    )
    print('\nESG Data Availability for Green Bond Issuers:')
    print(esg_coverage.to_string(index=False))
    
    # Calculate overall coverage
    total_bonds = esg_coverage['total_bonds'].sum()
    tier1_bonds = esg_coverage['bonds_with_complete_esg'].sum()
    tier2_bonds = esg_coverage['bonds_with_partial_esg'].sum()
    tier3_bonds = esg_coverage['bonds_with_no_esg'].sum()
    
    print(f'\nOverall Tier Distribution:')
    print(f'  Tier 1 (Complete ESG): {tier1_bonds} ({100*tier1_bonds/total_bonds:.1f}%)')
    print(f'  Tier 2 (Partial ESG): {tier2_bonds} ({100*tier2_bonds/total_bonds:.1f}%)')
    print(f'  Tier 3 (No ESG data): {tier3_bonds} ({100*tier3_bonds/total_bonds:.1f}%)')
except Exception as e:
    print(f'ESG coverage analysis skipped: {e}')


ESG DATA COVERAGE BY COUNTRY
ESG coverage analysis skipped: 'Issuer/Borrower Name Full'


In [18]:
# Summary statistics
print('\n' + '='*60)
print('PROCESSING SUMMARY')
print('='*60)
print(f'Final dataset shape: {merged_df.shape}')
print(f'Entities (firms): {merged_df["ric"].nunique()}')
print(f'Time periods: {merged_df["Year"].nunique()}')
print(f'Green bond issuers: {(merged_df["green_bond_issue"] == 1).sum()}')
print(f'Certified bonds: {(merged_df["is_certified"] == 1).sum()}')
print(f'\nMissing data:')
print(merged_df.isnull().sum().sum(), 'cells missing')
print(f'Missing rate: {merged_df.isnull().sum().sum() / (merged_df.shape[0] * merged_df.shape[1]) * 100:.2f}%')


PROCESSING SUMMARY
Final dataset shape: (19978, 65)
Entities (firms): 4004
Time periods: 5
Green bond issuers: 43
Certified bonds: 43

Missing data:
262535 cells missing
Missing rate: 20.22%


In [19]:
# Final trim to analysis period (remove the buffer year used for lags)
print(f'Trimming data to final analysis period: {START_YEAR} to {END_YEAR}')
final_df = merged_df[merged_df['Year'].between(START_YEAR, END_YEAR)].copy()
print(f'Final data shape: {final_df.shape}')

final_df.to_csv(config.PROCESSED_DATA_FILES["engineered"], index=False)
print(f"\n✓ Saved final engineered data to: {config.PROCESSED_DATA_FILES['engineered']}")

Trimming data to final analysis period: 2020 to 2024
Final data shape: (19978, 65)

✓ Saved final engineered data to: /Users/bunnypro/Projects/refinitiv-search/processed_data/final_engineered_panel_data.csv
